# Actividad: GraphRAG sobre el Congreso con MillenniumDB

**Duración estimada:** ~1 hora &nbsp;|&nbsp; **Modalidad:** individual o en parejas

En esta actividad vas a hacer el **retrieval** (recuperación) de un sistema *GraphRAG* sobre discursos del Congreso de Chile, usando la base de datos de grafos **MillenniumDB**, que además funciona como *vector store* (almacén de vectores con índice de similitud).

> ⚠️ **Importante:** en esta actividad **NO** vas a generar respuestas con un LLM. El foco está en el **retrieval** (qué documentos recupera el sistema) y en **evaluar** la calidad de esos resultados. La generación queda fuera de alcance.

## ¿Qué es GraphRAG?

RAG (*Retrieval-Augmented Generation*) clásico recupera documentos por **similitud vectorial**: se embebe la consulta, se buscan los vectores más cercanos y se devuelven los textos asociados.

**GraphRAG** añade la estructura del **grafo de conocimiento**: además de la similitud, se *recorre* el grafo para (a) traer metadatos (quién habló, de qué partido, en qué cámara, cuándo) y (b) **filtrar por patrones** (p. ej. *"intervenciones de parlamentarios independientes que hablen sobre X"*). Eso permite consultas mucho más ricas que la pura similitud.

## El grafo

La base ya está cargada con una **muestra** de los datos (≈400 discursos). El esquema tiene 5 tipos de nodos y 4 tipos de aristas:

| Nodo | Descripción | Propiedades clave |
|------|-------------|-------------------|
| `:Parlamentario` | Diputado/a o senador/a | `parl_id`, `nombre_completo`, `camara`, `fecha_nacimiento` |
| `:Partido` | Partido político | `nombre` |
| `:Unidad` | Pacto / unidad que representa | `nombre` |
| `:Participacion` | Un discurso / intervención | `fecha`, `camara`, `texto_principal` |
| `:Embedding` | Un vector (768-dim) de un *chunk* de una participación | `value` (tensor) |

```
(:Parlamentario)-[:enPartido]->(:Partido)
(:Parlamentario)-[:enUnidad]->(:Unidad)
(:Parlamentario)-[:enParticipacion]->(:Participacion)
(:Embedding)-[:tieneParticipation]->(:Participacion)
```

Nota: el **texto** NO se guarda en el nodo `:Embedding`, sino en `:Participacion`. Por eso, tras encontrar los vectores más parecidos, hay que **recorrer la arista** `tieneParticipation` para traer el texto. Además, un discurso largo puede tener **varios** embeddings (*chunks*), todos apuntando a la misma participación.

## 0. Preparación

Asumimos que el servidor de MillenniumDB ya está **levantado** con la base `congress.qm` cargada, y que nos conectamos vía el driver oficial de Python ([`millenniumdb-driver`](https://pypi.org/project/millenniumdb-driver/)).

Instala las dependencias si es la primera vez:

In [ ]:
# %pip install millenniumdb-driver sentence-transformers pandas numpy

In [17]:
import millenniumdb_driver
import pandas as pd
import numpy as np

# Ajusta la URL a la de tu servidor MillenniumDB (protocolo WebSocket).
MDB_URL = "ws://localhost:1234"

driver = millenniumdb_driver.driver(MDB_URL)


def run_df(query, columns=None):
    """Ejecuta una consulta MQL y devuelve un DataFrame de pandas.

    Si se entrega `columns`, renombra las columnas por POSICIÓN, en el mismo
    orden en que aparecen en el RETURN (así controlamos los nombres sin
    depender de cómo el servidor nombre cada proyección).
    """
    with driver.session() as session:
        result = session.run(query)
        df = result.to_df()
        if columns is not None and len(df.columns) == len(columns):
            df.columns = columns
        return df


# Prueba de conexión: 5 parlamentarios cualquiera
run_df("MATCH (?p :Parlamentario) RETURN ?p.nombre_completo LIMIT 5")

,p.nombre_completo
0,Francisco Huenchumilla Jaramillo
1,Rene Alinco Bustos
2,Rojo Edwards Silva
3,Pedro Araya Guerrero
4,Francisco Chahuan Chahuan


## 1. Explorar el grafo

Antes de buscar, conviene entender qué hay cargado. Contemos los nodos por tipo y miremos algunos ejemplos.

In [18]:
def contar(label):
    try:
        return int(run_df(f"MATCH (?n :{label}) RETURN COUNT(*)").iloc[0, 0])
    except Exception as e:
        return f"(no disponible: {e})"


for lbl in ["Parlamentario", "Partido", "Unidad", "Participacion", "Embedding"]:
    print(f"{lbl:15s}: {contar(lbl)}")

Parlamentario  : 160
Partido        : 20
Unidad         : 65
Participacion  : 400
Embedding      : 2492


In [19]:
# Algunos parlamentarios con su cámara
run_df(
    "MATCH (?p :Parlamentario) RETURN ?p.nombre_completo, ?p.camara LIMIT 8",
    columns=["nombre", "camara"],
)

,nombre,camara
0,Francisco Huenchumilla Jaramillo,Senado
1,Rene Alinco Bustos,C. Diputados
2,Rojo Edwards Silva,Senado
3,Pedro Araya Guerrero,Senado
4,Francisco Chahuan Chahuan,Senado
5,Carlos Bianchi Chelech,C. Diputados
6,Fidel Espinoza Sandoval,Senado
7,Gaston Von Muhlenbrock Zamora,C. Diputados


In [20]:
# ¿Qué partidos hay en la muestra?
run_df("MATCH (?pt :Partido) RETURN ?pt.nombre", columns=["partido"])

,partido
0,Partido Demócrata Cristiano
1,Partido Social Cristiano
2,Partido Renovación Nacional
3,Partido Socialista de Chile
4,Partido Unión Demócrata Independiente
5,Partido Por la Democracia
6,Federación Regionalista Verde Social
7,Partido Demócratas Chile
8,Partido Evolución Política (Evópoli)
9,Partido Comunista de Chile


## 2. Crear el índice de vectores (HNSW)

Para buscar por similitud, MillenniumDB necesita un **índice HNSW** sobre la propiedad `value` de los nodos `:Embedding`. Los vectores son de **768 dimensiones** y usamos **distancia coseno**.

> Si el índice ya existe, la celda simplemente avisará del error y puedes seguir.

In [21]:
INDEX_NAME = "idx_congreso"
DIM = 768  # multilingual-e5-base -> 768 dimensiones

crear_indice = f'''
CREATE HNSW INDEX "{INDEX_NAME}" WITH {{
  "property" = "value",
  "dimension" = {DIM},
  "maxEdges" = 8,
  "maxCandidates" = 16,
  "metric" = "cosineDistance"
}}
'''

try:
    with driver.session() as session:
      session.run(crear_indice)
      print("Índice HNSW creado.")
except Exception as e:
    print("Aviso (puede que el índice ya exista):", e)

Aviso (puede que el índice ya exista): Index "idx_congreso" already exists


## 3. El embedder: `multilingual-e5-base` (local)

Para embeber la **consulta** usamos el modelo [`intfloat/multilingual-e5-base`](https://huggingface.co/intfloat/multilingual-e5-base), que corre **localmente** (CPU está bien) y produce vectores de 768 dimensiones.

> 🔎 **Detalle clave de e5:** este modelo espera un **prefijo** en el texto: `"query: "` para consultas y `"passage: "` para documentos. Para que la búsqueda por similitud sea válida, el vector de la consulta debe vivir en el **mismo espacio** que los vectores almacenados. Los vectores de la base son **precomputados** (vienen de la Tarea original). Si tus resultados se ven pobres, prueba poniendo `USAR_PREFIJO_E5 = False`: es la principal palanca para alinear consulta y documentos.

In [22]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("intfloat/multilingual-e5-base")

USAR_PREFIJO_E5 = True  # <- experimenta con True/False y compara el retrieval


def embed_query(texto):
    entrada = ("query: " + texto) if USAR_PREFIJO_E5 else texto
    vec = embedder.encode(entrada, normalize_embeddings=True)
    return np.asarray(vec, dtype=np.float32)


def to_tensor_literal(vec):
    """Convierte un vector numpy al literal que entiende MQL: [v1, v2, ...]."""
    return "[" + ", ".join(f"{x:.8f}" for x in vec) + "]"


v = embed_query("litio")
print("dimensión del vector de consulta:", v.shape)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6687.71it/s]


dimensión del vector de consulta: (768,)


## 4. Búsqueda simple de similitud (RAG clásico)

El patrón básico de retrieval:
1. Embeber la pregunta → vector de consulta.
2. Pedirle al índice los `k` `:Embedding` más cercanos con `CALL HNSW_TOP_K(...)`.
3. **Recorrer** `tieneParticipation` para traer el `texto_principal` de la `:Participacion`.

La sintaxis de búsqueda kNN en MQL es:
```
CALL HNSW_TOP_K("<indice>", tensorFloat("[...]"), <k>, <candidatos>)
    YIELD ?object AS ?emb, ?distance
```
donde `?object` es el nodo `:Embedding` encontrado y `?distance` su **distancia coseno** (¡menor = más parecido!). `candidatos` (≥ k) controla el esfuerzo de búsqueda (recall).

In [43]:
def buscar_similares(pregunta, k=5, candidatos=100):
    """Retrieval clásico por similitud. Devuelve un DataFrame ordenado por
    cercanía, ya deduplicado a nivel de participación (un discurso largo tiene
    varios chunks que apuntan al mismo texto)."""
    qv = embed_query(pregunta)
    lit = to_tensor_literal(qv)
    # Pedimos más vectores de los necesarios (k*5) porque luego colapsamos
    # los chunks repetidos de un mismo discurso.
    query = f'''
    CALL HNSW_TOP_K("{INDEX_NAME}", tensorFloat("{lit}"), {k * 5}, {candidatos})
        YIELD ?object AS ?emb, ?distance
    MATCH (?parl :Parlamentario)-[:enParticipacion]->(?partic)<-[:tieneParticipation]-(?emb)
    ORDER BY ?distance
    RETURN ?distance, ?partic.fecha, ?partic.camara, ?parl.nombre_completo, ?partic.texto_principal
    '''
    df = run_df(query, columns=["distance", "fecha", "camara", "nombre", "texto"])
    df = (df.sort_values("distance")
            .drop_duplicates(subset=["texto"])
            .head(k)
            .reset_index(drop=True))
    df["similitud"] = 1 - df["distance"]
    return df


def mostrar(df, n_chars=300):
    for i, r in df.iterrows():
        extra = f" | {r['partido']}" if "partido" in df.columns else ""
        print(f"[{i+1}] sim={r['similitud']:.3f} | {r['nombre']} | {r['camara']} | {r['fecha']}{extra}")
        print("    ", str(r["texto"])[:n_chars].replace("\n", " "), "...")
        print()

In [44]:
df = buscar_similares("¿Qué se ha dicho en el congreso sobre el litio?", k=5)
mostrar(df)

[1] sim=0.862 | Viviana Delgado Riquelme | Cámara de Diputados | 2024-01-17Z
     El señor ROJAS (Prosecretario).- \n \nSolicitud de resolución Nº 991, de los diputados  Vlado Mirosevic ,   Alejandro Bernales , la diputada  Viviana Delgado , los diputados   Luis Malla  y   Sebastián Videla , que en su parte dispositiva señala:  \nLa Cámara de Diputadas y Diputados solicita a su e ...

[2] sim=0.840 | Cristian Tapia Ramos | Cámara de Diputados | 2023-12-19Z
     El señor CIFUENTES (Presidente).- \n \n Tiene la palabra el diputado Cristián Tapia .   \n \nEl señor TAPIA.- \n \n Señor Presidente, por su intermedio saludo a la ministra y a la subsecretaria de Minería.  \nCuando el Presidente anunció la Política Nacional del Litio, señalé que enhorabuena el Esta ...

[3] sim=0.839 | Karol Cariola Oliva | Cámara de Diputados | 2024-07-23Z
     El señor ROJAS (Prosecretario).- \n Solicitud de resolución N° 1.322, de los diputados Luis Alberto Cuello , María Candelaria Acevedo , Karol Cariola ,

Prueba con otra pregunta y observa los resultados (cambia el texto y el `k`):

In [45]:
df = buscar_similares("¿Qué se dice sobre las pensiones y las AFP?", k=5)
mostrar(df)

[1] sim=0.855 | Leonardo Soto Ferrada | Cámara de Diputados | 2024-01-24Z
     El señor CIFUENTES (Presidente).- \n \n Tiene la palabra el diputado Leonardo Soto .   \n \nEl señor SOTO (don Leonardo).- \n \n Señor Presidente, esta reforma previsional que hoy votaremos lleva catorce años en tabla en el Congreso Nacional.  \nHa habido varias iniciativas de varios Presidentes, y  ...

[2] sim=0.852 | Cristobal Urruticoechea Rios | Cámara de Diputados | 2024-01-24Z
     El señor CIFUENTES (Presidente).- \n \n Tiene la palabra el diputado Cristóbal Urruticoechea .   \n \nEl señor URRUTICOECHEA.- \n \n Señor Presidente, no estoy aquí para poner a las AFP por sobre los chilenos; estoy aquí para defender a los chilenos de los comunistas, de los  socialistas, de sus ali ...

[3] sim=0.847 | Carlos Bianchi Chelech | Cámara de Diputados | 2024-01-23Z
     El señor CIFUENTES (Presidente).- \n \n Tiene la palabra el diputado Carlos Bianchi .  \n \nEl señor BIANCHI.- \n \n Señor Presidente, después 

## 5. Búsquedas con patrones (GraphRAG)

Ahora aprovechamos el **grafo**. La idea: recuperar por similitud y, al mismo tiempo, **recorrer aristas** para enriquecer con metadatos del parlamentario y su partido, y luego **filtrar por patrones**.

Un parlamentario **independiente** es uno **sin** arista `:enPartido` (no tiene partido en los datos). Para clasificar fácilmente, primero traemos el mapa *parlamentario → partido* desde el grafo y luego lo usamos para filtrar los resultados del retrieval.

In [46]:
# Mapa parlamentario -> partido (los que NO aparezcan son independientes)
df_pt = run_df(
    "MATCH (?parl :Parlamentario)-[:enPartido]->(?pt :Partido) "
    "RETURN ?parl.nombre_completo, ?pt.nombre",
    columns=["nombre", "partido"],
)
partido_de = dict(zip(df_pt["nombre"], df_pt["partido"]))


def con_partido(df):
    df = df.copy()
    df["partido"] = df["nombre"].map(partido_de).fillna("(independiente / sin partido)")
    return df


print(f"{len(partido_de)} parlamentarios con partido en la muestra.")

121 parlamentarios con partido en la muestra.


### 5.1 Intervenciones de **independientes** que hablen sobre un tema

Recuperamos un conjunto más grande por similitud y nos quedamos con quienes **no tienen partido**.

In [47]:
df = con_partido(buscar_similares("seguridad pública y delincuencia", k=25))
indep = df[df["partido"].str.contains("independiente")].head(5).reset_index(drop=True)
mostrar(indep)

[1] sim=0.841 | Karim Bianchi Retamales | Senado | 2023-04-04Z | (independiente / sin partido)
     El señor BIANCHI.- \nMuchas gracias, Presidente. \nSaludo en las tribunas a todas y a todos, sin ideologías; no son enemigos, sino víctimas, y yo creo que en algún momento debemos reencontrarnos. \nYo me pregunto, si en Chile el principal tema hoy es la seguridad, cómo cresta no nos ponemos de acuer ...

[2] sim=0.833 | Leonidas Romero Saez | Cámara de Diputados | 2023-01-18Z | (independiente / sin partido)
     La señorita MIX,  doña Claudia  (Presidenta accidental).- \nTiene la palabra el diputado  Leonidas Romero .  \n \nEl señor ROMERO (don Leonidas).- \n \n Señora Presidenta, obviamente, me voy a sumar a las condolencias para la familia y para los colegas de la institución del comisario  Daniel Valdés  ...

[3] sim=0.828 | Kenneth Pugh Olavarria | Senado | 2023-03-15Z | (independiente / sin partido)
     El señor PUGH.- \nMuchas gracias, señor Presidente. \nHoy al mediodía, en Conce

### 5.2 Intervenciones de un **partido específico** sobre un tema

Por ejemplo, qué dice el **Frente Amplio** sobre la **permisología**:

In [48]:
df = con_partido(buscar_similares("permisología y tramitación de permisos sectoriales", k=30))
fa = df[df["partido"].str.contains("Frente Amplio", case=False, na=False)].head(5).reset_index(drop=True)
mostrar(fa)

[1] sim=0.826 | Jaime Saez Quiroz | Cámara de Diputados | 2023-05-15Z | Partido Frente Amplio
     -Diputado Sáez, don Jaime . Presentación de algún proyecto de conservación de infraestructura para acceder a los fondos de conservación de establecimientos educacionales y si ha postulado a los distintos fondos del Ministerio de Educación. (37723 de 11/05/2023). A Municipalidad de Chaitén.   \n-Dipu ...



### 5.3 (Bonus) Filtrar el patrón **dentro** de MQL

El filtro anterior se hizo en Python tras el retrieval. También se puede expresar el patrón **directamente en la consulta**, recorriendo hasta el nodo `:Partido`. Así, el grafo hace el trabajo de filtrar.

In [49]:
qv = embed_query("permisología y tramitación de permisos")
lit = to_tensor_literal(qv)
query = f'''
CALL HNSW_TOP_K("{INDEX_NAME}", tensorFloat("{lit}"), 50, 200) YIELD ?object AS ?emb, ?distance
MATCH (?parl :Parlamentario)-[:enParticipacion]->(?partic)<-[:tieneParticipation]-(?emb),
      (?parl)-[:enPartido]->(?pt :Partido)
WHERE ?pt.nombre == "Partido Frente Amplio"
ORDER BY ?distance
RETURN ?distance, ?parl.nombre_completo, ?pt.nombre, ?partic.texto_principal
LIMIT 5
'''
try:
    display(run_df(query, columns=["distance", "nombre", "partido", "texto"]))
except Exception as e:
    print("La consulta in-DB falló (usa el método de la sección 5.2):", e)

,distance,nombre,partido,texto
0,0.172103,Maria Francisca Bello Campos,Partido Frente Amplio,El señor CIFUENTES (Presidente).- \n \n Tiene ...
1,0.176597,Maite Orsini Pascal,Partido Frente Amplio,NORMATIVA SOBRE SEGURIDAD PRIVADA (TERCER TRÁM...
2,0.176638,Maite Orsini Pascal,Partido Frente Amplio,NORMATIVA SOBRE SEGURIDAD PRIVADA (TERCER TRÁM...
3,0.176646,Maite Orsini Pascal,Partido Frente Amplio,NORMATIVA SOBRE SEGURIDAD PRIVADA (TERCER TRÁM...
4,0.176769,Maite Orsini Pascal,Partido Frente Amplio,NORMATIVA SOBRE SEGURIDAD PRIVADA (TERCER TRÁM...


## 7. Ejercicios a entregar

Responde en este mismo notebook (código + breves comentarios en celdas markdown):

1. **Similitud simple.** Ejecuta `buscar_similares` para al menos **3** de estas preguntas y comenta la calidad de los resultados:
   - ¿Qué se ha dicho sobre el **litio**?
   - ¿Qué se dice sobre el **aborto**?
   - ¿Qué opinión hay sobre la **tenencia responsable de mascotas**?
   - ¿Cómo se habla de las **pensiones**?

2. **Patrones (GraphRAG).** Construye al menos **2** consultas que combinen similitud con un patrón del grafo. Ejemplos:
   - Intervenciones de **independientes** sobre un tema
   - Intervenciones de un **partido** sobre un tema 
   - Cómo se comparan las intervenciones de parlamentarios de diferentes unidades (ej: representante de Antofagasta vs de RM)

3. **Evaluación.** Para una pregunta a tu elección, calcula la **Precisión@10** (juicio manual). Luego **compara** el retrieval clásico vs. el filtrado por patrón (sección 6): ¿el patrón mejora la relevancia? ¿Por qué sí o por qué no?

4. **(Opcional) Tu propia pregunta.** Inventa una pregunta interesante y muestra su retrieval con y sin patrón.

In [50]:
# Cierra la conexión al terminar
driver.close()
print("Conexión cerrada.")

Conexión cerrada.
